In [1]:
import json
import re
import unicodedata
import pandas as pd

with open("../data/curated_papers_text.json", "r", encoding="utf-8") as file:
    data = json.load(file)

print("Dataset loaded successfully.")
print("Number of papers:", len(data))

Dataset loaded successfully.
Number of papers: 10


In [2]:
for paper in data[:2]:
    print("=" * 80)
    print("TITLE:", paper["title"])
    print("PAGE:", paper["pages"][0]["page_number"])
    print()
    print(paper["pages"][0]["text"][:1500])
    print()

TITLE: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
PAGE: 1

Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-
stream NLP tasks. However, their ability to access and precisely manipulate knowl-
edge is still limited, and hence on knowledge-intensive tasks, their performance
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentiable access mechanism to explicit non-p

In [3]:
def clean_text(text):
    if not text:
        return ""

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Fix common ligatures
    ligatures = {
        "ﬁ": "fi",
        "ﬂ": "fl",
        "ﬀ": "ff",
        "ﬃ": "ffi",
        "ﬄ": "ffl"
    }

    for old, new in ligatures.items():
        text = text.replace(old, new)

    # Join words split by hyphen + line break
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Replace remaining line breaks with spaces
    text = re.sub(r"\s*\n\s*", " ", text)

    # Remove repeated whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


sample_raw = data[0]["pages"][0]["text"]
sample_clean = clean_text(sample_raw)

print("RAW TEXT:")
print(sample_raw[:700])

print("\n" + "=" * 80)

print("CLEANED TEXT:")
print(sample_clean[:700])

RAW TEXT:
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research;‡University College London;⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-
stream NLP tasks. However, their ability to access and precisely manipulate knowl-
edge is still limited, and hence on knowledge-intensive tasks, their performance
lags behind task-s

CLEANED TEXT:
Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡, Ethan Perez⋆, Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†, Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela† †Facebook

In [4]:
processed_records = []

for paper in data:
    for page in paper["pages"]:
        raw_text = page.get("text", "")
        cleaned_text = clean_text(raw_text)

        processed_records.append({
            "paper_id": paper["id"],
            "title": paper["title"],
            "category": paper["category"],
            "pdf_url": paper["pdf_url"],
            "page_number": page["page_number"],
            "raw_text": raw_text,
            "cleaned_text": cleaned_text
        })

df_processed = pd.DataFrame(processed_records)

print("Total processed pages:", len(df_processed))
print("Empty cleaned pages:", (df_processed["cleaned_text"].str.len() == 0).sum())

df_processed.head()

Total processed pages: 218
Empty cleaned pages: 0


,paper_id,title,category,pdf_url,page_number,raw_text,cleaned_text
0,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,1,Retrieval-Augmented Generation for\nKnowledge-...,Retrieval-Augmented Generation for Knowledge-I...
1,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,2,The\tDivine\nComedy\t(x) q\nQuery\nEncoder\nq(...,The Divine Comedy (x) q Query Encoder q(x) MIP...
2,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,3,byθ that generates a current token based on a ...,byθ that generates a current token based on a ...
3,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,4,minimize the negative marginal log-likelihood ...,minimize the negative marginal log-likelihood ...
4,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,5,MSMARCO as an open-domain abstractive QA task....,MSMARCO as an open-domain abstractive QA task....


In [5]:
print("Total processed pages:", len(df_processed))
print("Empty cleaned pages:", (df_processed["cleaned_text"].str.len() == 0).sum())

Total processed pages: 218
Empty cleaned pages: 0


In [6]:
df_processed["raw_length"] = df_processed["raw_text"].str.len()
df_processed["cleaned_length"] = df_processed["cleaned_text"].str.len()
df_processed["characters_removed"] = (
    df_processed["raw_length"] - df_processed["cleaned_length"]
)

print("Average raw characters per page:",
      round(df_processed["raw_length"].mean(), 2))

print("Average cleaned characters per page:",
      round(df_processed["cleaned_length"].mean(), 2))

print("Total characters removed:",
      df_processed["characters_removed"].sum())

Average raw characters per page: 3669.31
Average cleaned characters per page: 3661.11
Total characters removed: 1787


In [7]:
import os

os.makedirs("../data/processed", exist_ok=True)

processed_data = df_processed[
    [
        "paper_id",
        "title",
        "category",
        "pdf_url",
        "page_number",
        "cleaned_text"
    ]
].to_dict(orient="records")

output_path = "../data/processed/cleaned_papers.json"

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(processed_data, file, ensure_ascii=False, indent=2)

print("Cleaned dataset saved successfully.")
print("Output file:", output_path)
print("Records saved:", len(processed_data))

Cleaned dataset saved successfully.
Output file: ../data/processed/cleaned_papers.json
Records saved: 218


In [8]:
with open("../data/processed/cleaned_papers.json", "r", encoding="utf-8") as file:
    verification_data = json.load(file)

print("Records loaded:", len(verification_data))
print("Fields:", list(verification_data[0].keys()))
print("Empty cleaned texts:",
      sum(1 for record in verification_data if not record["cleaned_text"].strip()))

Records loaded: 218
Fields: ['paper_id', 'title', 'category', 'pdf_url', 'page_number', 'cleaned_text']
Empty cleaned texts: 0


## Text Preprocessing Summary

The research paper text was cleaned and prepared for downstream RAG processing while preserving the original dataset.

### Preprocessing Steps

- Loaded all 10 research papers and 218 extracted pages.
- Normalized Unicode characters and common ligatures.
- Rejoined words that were split across PDF line breaks.
- Replaced unnecessary line breaks with spaces.
- Normalized repeated whitespace.
- Preserved important metadata including paper ID, title, category, PDF URL, and page number.
- Retained technical symbols and research content to avoid overly aggressive cleaning.
- Saved the cleaned text as a separate processed dataset without modifying the original source data.

### Validation Results

- 218 pages were successfully processed.
- No cleaned pages were empty.
- Average raw page length: 3,669.31 characters.
- Average cleaned page length: 3,661.11 characters.
- 1,787 formatting characters were removed during preprocessing.

### Output

The cleaned dataset was saved to:

`data/processed/cleaned_papers.json`

This processed dataset is ready for the next stage of the RAG pipeline, including text chunking and embedding generation.